# Binning and binarization
Rebuilt for scikit-learn 1.9 and pandas 3. Data: `data/titanic_train.csv` (the Kaggle Titanic training file, 891 rows).

In [ ]:
# libraries for data, splitting, the model, scoring and the two transformers
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import KBinsDiscretizer, Binarizer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline

## 1. Discretization with KBinsDiscretizer

In [ ]:
# keep three columns and drop the 177 rows with a missing age
df = pd.read_csv("data/titanic_train.csv", usecols=["Age", "Fare", "Survived"])
df = df.dropna()
print(df.shape)
df.head()

In [ ]:
# inputs and target, then an 80/20 split
X = df[["Age", "Fare"]]
y = df["Survived"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# baseline: a decision tree on the raw numbers
# random_state=0 fixes how the tree breaks ties, so the numbers repeat
clf = DecisionTreeClassifier(random_state=0)
clf.fit(X_train, y_train)
print("test accuracy:", accuracy_score(y_test, clf.predict(X_test)))
print("10-fold CV:", cross_val_score(DecisionTreeClassifier(random_state=0), X, y, cv=10).mean())

In [ ]:
# one discretizer per column: 15 equal-frequency bins, output = bin number
kbin_age = KBinsDiscretizer(n_bins=15, encode="ordinal", strategy="quantile")
kbin_fare = KBinsDiscretizer(n_bins=15, encode="ordinal", strategy="quantile")
trf = ColumnTransformer([("age", kbin_age, ["Age"]), ("fare", kbin_fare, ["Fare"])])

# learn the edges on the training set only, then apply them to both sets
X_train_trf = trf.fit_transform(X_train)
X_test_trf = trf.transform(X_test)

In [ ]:
# the learned edges: 16 edges make 15 bins
print(trf.named_transformers_["age"].bin_edges_[0])
print(trf.named_transformers_["fare"].bin_edges_[0])

In [ ]:
# each value, its bin number, and the range of that bin
# KBinsDiscretizer's bins include their left edge, so right=False makes pd.cut agree
age_edges = trf.named_transformers_["age"].bin_edges_[0]
fare_edges = trf.named_transformers_["fare"].bin_edges_[0]
output = pd.DataFrame({
    "age": X_train["Age"], "age_trf": X_train_trf[:, 0],
    "age_range": pd.cut(X_train["Age"], age_edges, right=False),
    "fare": X_train["Fare"], "fare_trf": X_train_trf[:, 1],
    "fare_range": pd.cut(X_train["Fare"], fare_edges, right=False),
})
output.sample(5, random_state=1)

In [ ]:
# the same tree on the bin numbers
clf = DecisionTreeClassifier(random_state=0)
clf.fit(X_train_trf, y_train)
print("test accuracy:", accuracy_score(y_test, clf.predict(X_test_trf)))

# cross-validation: binning must go inside a pipeline,
# so each fold learns its edges from its own training part
pipe = make_pipeline(trf, DecisionTreeClassifier(random_state=0))
print("10-fold CV:", cross_val_score(pipe, X, y, cv=10).mean())

## 2. Trying every strategy and number of bins

In [ ]:
# one function: bin both columns, report the CV accuracy, plot before and after
def discretize(bins, strategy):
    trf = ColumnTransformer([
        ("age", KBinsDiscretizer(n_bins=bins, encode="ordinal", strategy=strategy), ["Age"]),
        ("fare", KBinsDiscretizer(n_bins=bins, encode="ordinal", strategy=strategy), ["Fare"]),
    ])
    pipe = make_pipeline(trf, DecisionTreeClassifier(random_state=0))
    print(strategy, bins, "CV accuracy:", round(cross_val_score(pipe, X, y, cv=10).mean(), 4))

    X_trf = trf.fit_transform(X)
    fig = make_subplots(2, 2, subplot_titles=["Age before", "Age after", "Fare before", "Fare after"])
    fig.add_histogram(x=X["Age"], row=1, col=1)
    fig.add_histogram(x=X_trf[:, 0], marker_color="#E45756", row=1, col=2)
    fig.add_histogram(x=X["Fare"], row=2, col=1)
    fig.add_histogram(x=X_trf[:, 1], marker_color="#E45756", row=2, col=2)
    fig.update_layout(showlegend=False, height=550, title=f"{strategy}, {bins} bins")
    fig.show()

discretize(10, "quantile")

In [ ]:
discretize(10, "uniform")

In [ ]:
discretize(5, "kmeans")

In [ ]:
# all three strategies side by side, 5 bins on the training Fare column: edges and rows per bin
for s in ["uniform", "quantile", "kmeans"]:
    kb = KBinsDiscretizer(n_bins=5, encode="ordinal", strategy=s).fit(X_train[["Fare"]])
    counts = np.bincount(kb.transform(X_train[["Fare"]])[:, 0].astype(int), minlength=5)
    print(f"{s:9s} edges {np.round(kb.bin_edges_[0], 1)}  rows per bin {counts}")

In [ ]:
# a table of CV accuracies for every strategy and number of bins
rows = []
for s in ["uniform", "quantile", "kmeans"]:
    for n in [5, 10, 15]:
        trf = ColumnTransformer([
            ("age", KBinsDiscretizer(n_bins=n, encode="ordinal", strategy=s), ["Age"]),
            ("fare", KBinsDiscretizer(n_bins=n, encode="ordinal", strategy=s), ["Fare"]),
        ])
        pipe = make_pipeline(trf, DecisionTreeClassifier(random_state=0))
        rows.append((s, n, cross_val_score(pipe, X, y, cv=10).mean()))
pd.DataFrame(rows, columns=["strategy", "bins", "cv_accuracy"]).pivot(index="bins", columns="strategy").round(3)

## 3. Custom binning with domain knowledge

In [ ]:
# our own edges: child below 18, adult 18 to 59, senior 60 and over
age_group = pd.cut(df["Age"], bins=[0, 18, 60, np.inf], right=False,
                   labels=["child", "adult", "senior"])
pd.concat([df["Age"], age_group.rename("age_group")], axis=1).head()

In [ ]:
# survival rate per group
df.groupby(age_group, observed=True)["Survived"].agg(["mean", "size"])

## 4. Binarization

In [ ]:
# family = siblings/spouses + parents/children on board
df2 = pd.read_csv("data/titanic_train.csv", usecols=["Age", "Fare", "SibSp", "Parch", "Survived"]).dropna()
df2["family"] = df2["SibSp"] + df2["Parch"]
df2 = df2.drop(columns=["SibSp", "Parch"])
X2 = df2.drop(columns=["Survived"])
y2 = df2["Survived"]
X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y2, test_size=0.2, random_state=42)
X2_train.head()

In [ ]:
# baseline without binarization
clf = DecisionTreeClassifier(random_state=0)
clf.fit(X2_train, y2_train)
print("test accuracy:", accuracy_score(y2_test, clf.predict(X2_test)))
print("10-fold CV:", cross_val_score(DecisionTreeClassifier(random_state=0), X2, y2, cv=10).mean())

In [ ]:
# family > 0 becomes 1 (travelling with family), family = 0 stays 0 (alone)
trf = ColumnTransformer([("bin", Binarizer(threshold=0), ["family"])], remainder="passthrough")
trf.set_output(transform="pandas")
X2_train_trf = trf.fit_transform(X2_train)
X2_test_trf = trf.transform(X2_test)
X2_train_trf.head()

In [ ]:
# the same tree on the binarized data
clf = DecisionTreeClassifier(random_state=0)
clf.fit(X2_train_trf, y2_train)
print("test accuracy:", accuracy_score(y2_test, clf.predict(X2_test_trf)))
pipe = make_pipeline(trf, DecisionTreeClassifier(random_state=0))
print("10-fold CV:", cross_val_score(pipe, X2, y2, cv=10).mean())

In [ ]:
# survival rate: alone (0) against with family (1)
df2.groupby(df2["family"] > 0)["Survived"].agg(["mean", "size"])

In [ ]:
# a value equal to the threshold becomes 0: only values strictly above it become 1
Binarizer(threshold=127.5).fit_transform([[0, 100, 127.5, 128, 255]])